# EDSS `0101` 미연결 학교연도 키 진단

## tl;dr

데이터셋별 미연결 316건은 중복 제거 후 76개 고유 `(연도, 개방ID)`다. 영향은 8,106행(전체의 0.0594%)이며, 외부 교차검증 전에는 삭제하거나 강제 매핑하지 않는다.

## Context & Methods

이 노트북은 `scripts/diagnose_edss_orphan_keys.py`가 생성한 키 수준 산출물을 검토한다. 원본 패널의 grain은 보존하며 취업 제한 패널의 개인 수준 필드는 노출하지 않는다.

### Key Assumptions

시간적 분류는 관측범위 차이이며 신설·폐교·통폐합 또는 ID 변경의 증명은 아니다.

In [1]:
import csv
import json
from pathlib import Path

repo = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
summary_path = repo / 'data/metadata/edss_orphan_key_diagnosis.json'
keys_path = repo / 'data/metadata/edss_orphan_school_year_keys.csv'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
with keys_path.open(encoding='utf-8-sig', newline='') as handle:
    orphan_keys = list(csv.DictReader(handle))
len(orphan_keys)


76

## Data

In [2]:
core = {
    'dataset_occurrences': summary['dataset_orphan_key_occurrence_count'],
    'distinct_keys': summary['distinct_orphan_school_year_key_count'],
    'distinct_open_ids': summary['distinct_orphan_open_id_count'],
    'affected_rows': summary['affected_row_count'],
}
core

{'dataset_occurrences': 316,
 'distinct_keys': 76,
 'distinct_open_ids': 69,
 'affected_rows': 8106}

## Results

In [3]:
classification_table = [
    {
        'classification': name,
        'distinct_keys': count,
        'dataset_occurrences': summary['classification_dataset_occurrence_counts'][name],
        'affected_rows': summary['classification_affected_row_counts'][name],
    }
    for name, count in summary['classification_key_counts'].items()
]
classification_table

[{'classification': 'after_base_last_seen',
  'distinct_keys': 32,
  'dataset_occurrences': 48,
  'affected_rows': 4289},
 {'classification': 'before_base_first_seen',
  'distinct_keys': 38,
  'dataset_occurrences': 261,
  'affected_rows': 3807},
 {'classification': 'internal_base_gap',
  'distinct_keys': 1,
  'dataset_occurrences': 2,
  'affected_rows': 2},
 {'classification': 'open_id_absent_all_years',
  'distinct_keys': 5,
  'dataset_occurrences': 5,
  'affected_rows': 8}]

In [4]:
dataset_table = [row for row in summary['datasets'] if row['orphan_key_count']]
dataset_table

[{'catalog_code': '0305',
  'dataset': '신입생충원현황',
  'access_tier': 'panel',
  'orphan_key_count': 32,
  'affected_row_count': 262},
 {'catalog_code': '0310',
  'dataset': '재적학생현황',
  'access_tier': 'panel',
  'orphan_key_count': 33,
  'affected_row_count': 273},
 {'catalog_code': '0316',
  'dataset': '중도탈락학생현황',
  'access_tier': 'panel',
  'orphan_key_count': 48,
  'affected_row_count': 907},
 {'catalog_code': '0502',
  'dataset': '전체교원대비전임교원현황',
  'access_tier': 'panel',
  'orphan_key_count': 15,
  'affected_row_count': 85},
 {'catalog_code': '0503',
  'dataset': '전임교원1인당학생현황',
  'access_tier': 'panel',
  'orphan_key_count': 9,
  'affected_row_count': 38},
 {'catalog_code': '0601',
  'dataset': '국내외학술지게재논문실적',
  'access_tier': 'panel',
  'orphan_key_count': 32,
  'affected_row_count': 251},
 {'catalog_code': '0714',
  'dataset': '등록금현황',
  'access_tier': 'panel',
  'orphan_key_count': 32,
  'affected_row_count': 1806},
 {'catalog_code': '1014',
  'dataset': '연구비수혜실적',
  'access_tier':

In [5]:
priority_review = [
    row for row in orphan_keys
    if row['classification'] in {'internal_base_gap', 'open_id_absent_all_years'}
]
priority_review

[{'year': '2009',
  'open_id': '4124740779',
  'classification': 'open_id_absent_all_years',
  'base_first_year': '',
  'base_last_year': '',
  'nearest_base_year_before': '',
  'nearest_base_year_after': '',
  'base_school_types': '',
  'base_regions': '',
  'base_branch_types': '',
  'orphan_school_types': '대학원',
  'dataset_count': '1',
  'datasets': '0502 전체교원대비전임교원현황',
  'dataset_orphan_key_occurrences': '1',
  'affected_row_count': '1'},
 {'year': '2009',
  'open_id': '5814962685',
  'classification': 'open_id_absent_all_years',
  'base_first_year': '',
  'base_last_year': '',
  'nearest_base_year_before': '',
  'nearest_base_year_after': '',
  'base_school_types': '',
  'base_regions': '',
  'base_branch_types': '',
  'orphan_school_types': '대학원',
  'dataset_count': '1',
  'datasets': '0502 전체교원대비전임교원현황',
  'dataset_orphan_key_occurrences': '1',
  'affected_row_count': '1'},
 {'year': '2009',
  'open_id': '6566148042',
  'classification': 'open_id_absent_all_years',
  'base_first

## Takeaways

- 316은 데이터셋별 발생 합계이고 실제 고유 키는 76개다.
- 최초 등장 전·마지막 등장 후 키가 70개로 대부분을 차지한다.
- 전 기간 미등장 5개와 내부 공백 1개를 외부 학교명·상태 교차표로 먼저 검증한다.
- 검증 전 결합은 left join과 명시적 미연결 플래그를 사용한다.